# Lab 20-01 · Transconductance Efficiency, Speed and Gain

**Predict first**, then run. The handout is [README.md](README.md). All numbers use the chapter's illustrative T130 process.

In [1]:
import math
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import gmid as g

p = g.Process()
print(f"V_T = {p.vt * 1e3:.2f} mV, weak-inversion limit 1/(n V_T) = {1 / (p.n * p.vt):.2f} S/A")
for L in (0.15e-6, 0.5e-6, 1e-6):
    print(f"L = {L * 1e6:.2f} um: lambda_c = {p.lam(L):.4f}")

V_T = 25.85 mV, weak-inversion limit 1/(n V_T) = 29.76 S/A
L = 0.15 um: lambda_c = 0.0754
L = 0.50 um: lambda_c = 0.0226
L = 1.00 um: lambda_c = 0.0113


## Part 1 · Efficiency from weak to strong inversion

*Predict* g_m/I_D at IC = 0.01, 1 and 100, and the square law's 2/V_ov at the same overdrive.

In [2]:
ics = np.logspace(-3, 3, 241)
long_ = [g.gm_over_id(ic, p.n, p.vt) for ic in ics]
short = [g.gm_over_id(ic, p.n, p.vt, p.lam(0.15e-6)) for ic in ics]
sq = [2 / g.vov_of_ic(ic, p.n, p.vt) if g.vov_of_ic(ic, p.n, p.vt) > 0 else np.nan for ic in ics]
for ic in (0.01, 1, 100):
    print(f"IC = {ic:g}: long channel {g.gm_over_id(ic, p.n, p.vt):.2f} S/A, 0.15 um {g.gm_over_id(ic, p.n, p.vt, p.lam(0.15e-6)):.2f} S/A")
print(f"the square law exceeds 1/(n V_T) below V_ov = 2 n V_T = {2 * p.n * p.vt * 1e3:.1f} mV")
plt.figure(); plt.loglog(ics, long_, label="charge-based, long channel"); plt.loglog(ics, short, label="with velocity saturation, 0.15 um")
plt.loglog(ics, sq, "--", label="square law 2/V_ov"); plt.axhline(1 / (p.n * p.vt), color="k", lw=0.8)
plt.ylim(0.5, 200); plt.xlabel("inversion coefficient IC"); plt.ylabel("g_m/I_D (S/A)"); plt.legend(); plt.show()

IC = 0.01: long channel 29.46 S/A, 0.15 um 29.44 S/A
IC = 1: long channel 18.39 S/A, 0.15 um 17.55 S/A
IC = 100: long channel 2.83 S/A, 0.15 um 1.48 S/A
the square law exceeds 1/(n V_T) below V_ov = 2 n V_T = 67.2 mV


## Part 2 · The design chart

*Predict* which curves move with L and which do not. Then re-size Example 20-01.2 from a lookup table.

In [3]:
targets = np.linspace(3, 28, 51)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
table = {}
for L in (0.15e-6, 0.5e-6, 1e-6):
    rows = []
    for t in targets:
        ic = g.ic_for_gm_over_id(t, p, L)
        rows.append((t, g.id_per_w(ic, p, L), g.ft(ic, p, L), g.intrinsic_gain(ic, p, L)))
    rows = np.array(rows)
    table[L] = rows
    lab = f"L = {L * 1e6:.2f} um"
    ax[0].semilogy(rows[:, 0], rows[:, 1], label=lab); ax[1].semilogy(rows[:, 0], rows[:, 2] / 1e9, label=lab); ax[2].plot(rows[:, 0], rows[:, 3], label=lab)
ax[0].set_ylabel("I_D/W (A/m = uA/um)"); ax[1].set_ylabel("f_T (GHz)"); ax[2].set_ylabel("intrinsic gain")
for a in ax:
    a.set_xlabel("g_m/I_D (S/A)"); a.legend()
plt.tight_layout(); plt.show()
# lookup-table sizing of Example 20-01.2: interpolate the 0.5 um current density at 15 S/A
rows = table[0.5e-6]
j15 = np.interp(15, rows[:, 0], rows[:, 1])
gm = 2 * math.pi * 100e6 * 2e-12
print(f"I_D = {gm / 15 * 1e6:.1f} uA, current density {j15:.3f} uA/um, W = {gm / 15 / j15 * 1e6:.1f} um")

I_D = 83.8 uA, current density 1.913 uA/um, W = 43.8 um


## Part 3 · The efficiency-speed sweet spot

*Predict* whether the peak of (g_m/I_D) f_T moves to weaker or stronger inversion as L grows.

In [4]:
from scipy.optimize import minimize_scalar
plt.figure()
for L in (0.15e-6, 0.5e-6, 1e-6):
    prod = [g.gm_over_id(ic, p.n, p.vt, p.lam(L)) * g.ft(ic, p, L) / 1e9 for ic in ics]
    plt.loglog(ics, prod, label=f"L = {L * 1e6:.2f} um")
    r = minimize_scalar(lambda x: -g.gm_over_id(10 ** x, p.n, p.vt, p.lam(L)) * g.ft(10 ** x, p, L), bounds=(-3, 4), method="bounded")
    ic = 10 ** r.x
    print(f"L = {L * 1e6:.2f} um: peak at IC = {ic:.2f}, g_m/I_D = {g.gm_over_id(ic, p.n, p.vt, p.lam(L)):.2f} S/A, f_T = {g.ft(ic, p, L) / 1e9:.2f} GHz")
plt.xlabel("IC"); plt.ylabel("(g_m/I_D) f_T (GHz/V)"); plt.legend(); plt.show()

L = 0.15 um: peak at IC = 6.86, g_m/I_D = 8.00 S/A, f_T = 35.38 GHz
L = 0.50 um: peak at IC = 22.59, g_m/I_D = 5.12 S/A, f_T = 6.72 GHz
L = 1.00 um: peak at IC = 44.94, g_m/I_D = 3.84 S/A, f_T = 2.51 GHz


## Part 4 · Corners and temperature

*Predict* how much the current of Example 20-01.1's transistor changes for a 50 mV threshold shift with the gate voltage fixed, and how g_m changes with temperature at fixed current.

In [5]:
W, L, ID = 10e-6, 0.5e-6, 20e-6
vgs = p.vth + g.vov_of_ic(ID / p.ispec(W, L), p.n, p.vt, p.lam(L))
shifts = np.linspace(-0.08, 0.08, 33)
cur = [g.drain_current(vgs, g.Process(vth=p.vth + d), W, L) for d in shifts]
for d in (0.05, -0.05):
    print(f"threshold {d * 1e3:+.0f} mV with V_GS fixed at {vgs:.4f} V: I_D = {g.drain_current(vgs, g.Process(vth=p.vth + d), W, L) * 1e6:.2f} uA")
temps = np.linspace(233.15, 398.15, 34)
gms = []
for T in temps:
    pt = g.Process(temp=T, mu=p.mu * (T / 300) ** -1.5)
    gms.append(ID * g.gm_over_id(ID / pt.ispec(W, L), pt.n, pt.vt, pt.lam(L)))
print(f"g_m at fixed current: {gms[0] * 1e6:.1f} uS at -40 C, {gms[-1] * 1e6:.1f} uS at 125 C")
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].semilogy(shifts * 1e3, np.array(cur) * 1e6); ax[0].set_xlabel("threshold shift (mV)"); ax[0].set_ylabel("I_D at fixed V_GS (uA)")
ax[1].plot(temps - 273.15, np.array(gms) * 1e6); ax[1].set_xlabel("temperature (C)"); ax[1].set_ylabel("g_m at fixed I_D (uS)")
plt.tight_layout(); plt.show()

threshold +50 mV with V_GS fixed at 0.5157 V: I_D = 8.61 uA
threshold -50 mV with V_GS fixed at 0.5157 V: I_D = 38.48 uA
g_m at fixed current: 362.6 uS at -40 C, 234.2 uS at 125 C


## Part 5 · Intrinsic gain in SPICE

*Predict* the gain of `gain.cir` from g_m/I_D and V_A = (1 + lambda V_DS)/lambda for each LAMBDA, then run ngspice.

In [6]:
from scipy.optimize import brentq
KP, VTO, WL, I = 200e-6, 0.45, 10.0, 20e-6
for lam in (0.1, 0.05):
    vov = brentq(lambda x: KP / 2 * WL * x ** 2 * (1 + lam * (VTO + x)) - I, 1e-6, 2)
    vds = VTO + vov
    gm_ = KP * WL * vov * (1 + lam * vds)
    va = (1 + lam * vds) / lam
    print(f"LAMBDA = {lam}: g_m/I_D = {gm_ / I:.2f} S/A, V_A = {va:.2f} V, predicted gain {gm_ / I * va:.1f}")
try:
    out = subprocess.run(["ngspice", "-b", "gain.cir"], capture_output=True, text=True, timeout=60).stdout
    print("\n".join(line for line in out.splitlines() if line.strip().startswith("gain")))
except FileNotFoundError:
    print("ngspice not installed; the README lists the expected values")

LAMBDA = 0.1: g_m/I_D = 14.55 S/A, V_A = 10.59 V, predicted gain 154.1
LAMBDA = 0.05: g_m/I_D = 14.35 S/A, V_A = 20.59 V, predicted gain 295.4
gaina               =  1.539816e+02
gainb               =  2.951318e+02


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).